In [21]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, root_mean_squared_error
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')


In [64]:
train_df = pd.read_csv("trainpaper.csv")
test_df = pd.read_csv("testapepr.csv")

In [65]:
train_df.head()

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,remarks,timespan,text,evaluation
0,92_4,92,accept,2.0,1,es,NaN,2014-07-05,Me parece un artículo interesante que pudo cen...,1
1,106_2,106,accept,3.0,1,es,"El paper es muy bueno, pero no es original, ya...",2014-07-05,Muy buen artículo para comprender la ubicación...,1
2,88_2,88,accept,3.0,0,es,La idea es buena pero debe arreglar lo que se ...,2013-07-05,Correcciones de Formato: 1.\tEn palabras clav...,1
3,160_2,160,accept,4.0,1,es,NaN,2015-07-05,"El artículo es interesante, abarca en profundi...",2
4,139_1,139,accept,4.0,0,es,NaN,2015-07-05,Los autores proponen el análisis de los datos ...,1


In [66]:
pd.crosstab(train_df["preliminary_decision"], train_df["evaluation"])

evaluation,-2,-1,0,1,2
preliminary_decision,,,,,
accept,12,20,37,61,82
no decision,0,0,0,1,0
probably reject,5,3,3,4,1
reject,52,24,11,4,4


In [67]:
for col in train_df.columns:
    print(col, train_df[col].unique()[:200])

review_id <ArrowStringArray>
[ '92_4', '106_2',  '88_2', '160_2', '139_1',  '41_3',  '73_2',  '50_4',
 '170_1',  '40_1',
 ...
  '18_1',  '29_3',  '98_3',  '33_2', '113_2',  '42_3', '143_1', '141_2',
 '128_2',  '11_2']
Length: 200, dtype: str
paper_id [ 92 106  88 160 139  41  73  50 170  40  20  39  21 166  80  67  38  12
  35 149 113   8  70 116 144  66  79 167 129  19  15  57  17  37 165 128
 122  68 125  24  82 112  62 101  63 121 136 100 123  76 132 161  25  33
  97 124 140 134 130 145  44 104 118   9   5  26  47 120 115 147 127 107
   1 172  89 135 109  11 138  65  98   3 114  81  71 102  49  14  72  29
 164 148 126  32 162  18  28 137  74  56 153 103   7  43  22 169  34  83
  23  31  93  95   2 171 111  55 133 158 110  91   6  10 108 151  45 156
  42 143 141 142  59  30 159 154  78   4  53  69 157  75 131  58  90  16
 163 155  99  77  96  85 119  84 168 152 117  60  86  52  48  94 150  27
  87]
preliminary_decision <ArrowStringArray>
['accept', 'probably reject', 'reject', 'no de

In [68]:
train_df.replace({"es":1, "en":0, "reject":0, "probably reject": 1, "no decision":2, "accept":3}, inplace=True)

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,remarks,timespan,text,evaluation
0,92_4,92,3,2.0,1,1,NaN,2014-07-05,Me parece un artículo interesante que pudo cen...,1
1,106_2,106,3,3.0,1,1,"El paper es muy bueno, pero no es original, ya...",2014-07-05,Muy buen artículo para comprender la ubicación...,1
2,88_2,88,3,3.0,0,1,La idea es buena pero debe arreglar lo que se ...,2013-07-05,Correcciones de Formato: 1.\tEn palabras clav...,1
3,160_2,160,3,4.0,1,1,NaN,2015-07-05,"El artículo es interesante, abarca en profundi...",2
4,139_1,139,3,4.0,0,1,NaN,2015-07-05,Los autores proponen el análisis de los datos ...,1
...,...,...,...,...,...,...,...,...,...,...
319,167_3,167,0,4.0,-2,1,NaN,2015-07-05,"El formato simplemente pésimo, la redacción de...",-2
320,87_2,87,3,3.0,1,1,Mi duda es si está en el contexto de este evento,2013-07-05,"Se propone un modelo CityCA, basado en autómat...",2
321,16_1,16,3,3.0,0,1,NaN,2010-07-05,NaN,0
322,150_1,150,3,3.0,0,1,NaN,2015-07-05,Este artículo describe los resultados de simul...,0


In [69]:
def returntime(X):
    y,m,d = X.split("-")
    return int(d)+30*int(m)+365*int(y)
def returnrev(X):
    pap, rev = X.split("_")
    return int(rev)

In [70]:
train_df["review_id"]= train_df["review_id"].apply(returnrev)
train_df["timespan"] = train_df["timespan"].apply(returntime)
train_df

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,remarks,timespan,text,evaluation
0,4,92,3,2.0,1,1,NaN,735325,Me parece un artículo interesante que pudo cen...,1
1,2,106,3,3.0,1,1,"El paper es muy bueno, pero no es original, ya...",735325,Muy buen artículo para comprender la ubicación...,1
2,2,88,3,3.0,0,1,La idea es buena pero debe arreglar lo que se ...,734960,Correcciones de Formato: 1.\tEn palabras clav...,1
3,2,160,3,4.0,1,1,NaN,735690,"El artículo es interesante, abarca en profundi...",2
4,1,139,3,4.0,0,1,NaN,735690,Los autores proponen el análisis de los datos ...,1
...,...,...,...,...,...,...,...,...,...,...
319,3,167,0,4.0,-2,1,NaN,735690,"El formato simplemente pésimo, la redacción de...",-2
320,2,87,3,3.0,1,1,Mi duda es si está en el contexto de este evento,734960,"Se propone un modelo CityCA, basado en autómat...",2
321,1,16,3,3.0,0,1,NaN,733865,NaN,0
322,1,150,3,3.0,0,1,NaN,735690,Este artículo describe los resultados de simul...,0


In [71]:
modifieddf = train_df.copy()

In [72]:
def changeremarks(X):
    if pd.isna(X):
        return 0
    else:
        return 1
modifieddf["remarks_stat"] = modifieddf["remarks"].apply(changeremarks)
modifieddf.drop(["remarks"], axis=1)
modifieddf.head()

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,remarks,timespan,text,evaluation,remarks_stat
0,4,92,3,2.0,1,1,NaN,735325,Me parece un artículo interesante que pudo cen...,1,0
1,2,106,3,3.0,1,1,"El paper es muy bueno, pero no es original, ya...",735325,Muy buen artículo para comprender la ubicación...,1,1
2,2,88,3,3.0,0,1,La idea es buena pero debe arreglar lo que se ...,734960,Correcciones de Formato: 1.\tEn palabras clav...,1,1
3,2,160,3,4.0,1,1,NaN,735690,"El artículo es interesante, abarca en profundi...",2,0
4,1,139,3,4.0,0,1,NaN,735690,Los autores proponen el análisis de los datos ...,1,0


In [73]:
train_df.describe()

,review_id,paper_id,confidence,orientation,timespan,evaluation
count,324.000000,324.000000,322.000000,324.000000,324.000000,324.000000
mean,1.854938,85.259259,3.555901,-0.237654,734905.925926,0.182099
std,0.829379,49.786797,0.856519,1.027035,701.441731,1.503614
min,1.000000,1.000000,1.000000,-2.000000,733865.000000,-2.000000
25%,1.000000,38.750000,3.000000,-1.000000,733865.000000,-1.000000
50%,2.000000,91.500000,4.000000,0.000000,735325.000000,0.000000
75%,2.000000,127.000000,4.000000,1.000000,735325.000000,2.000000
max,4.000000,172.000000,5.000000,2.000000,735690.000000,2.000000


In [74]:
scaler = StandardScaler()
onehot = OneHotEncoder(sparse_output=False)
impute = SimpleImputer()

In [75]:
def impp(X):
    if pd.isna(X):
        return -1
    else:
        return X
modifieddf["confidence"]=modifieddf["confidence"].apply(impp)

In [76]:
x_train, x_val, y_train,y_val = train_test_split(modifieddf.drop(["text", "evaluation", "remarks"], axis=1), modifieddf["evaluation"], stratify=modifieddf["evaluation"], test_size=0.2)

In [77]:
x_train

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,timespan,remarks_stat
69,2,44,3,5.0,-1,1,734960,0
107,2,81,3,3.0,-1,1,734960,0
219,1,125,0,3.0,-2,1,735325,0
173,1,10,3,3.0,1,1,733865,0
37,1,165,3,4.0,0,1,735690,0
...,...,...,...,...,...,...,...,...
267,1,84,3,3.0,1,1,734960,0
8,1,170,0,5.0,-2,1,735690,0
195,3,42,1,2.0,-1,1,733865,0
84,2,107,0,4.0,0,1,735325,0


In [78]:
x_train.isna().sum()

review_id               0
paper_id                0
preliminary_decision    0
confidence              0
orientation             0
lan                     0
timespan                0
remarks_stat            0
dtype: int64

In [79]:
corr_col =["paper_id", "confidence", "orientation", "evaluation"]
numerical_col =["paper_id", "confidence", "orientation"]

In [80]:
num = scaler.fit_transform(x_train[["timespan", "paper_id"]])

In [91]:
x_out = np.hstack((num, x_train.drop(["timespan", "paper_id"], axis=1)))

In [82]:
num1 = scaler.transform(x_val[["timespan", "paper_id"]])
val_out = np.hstack((num1, x_val.drop(["timespan", "paper_id"], axis=1) ))

In [83]:
test_df.replace({"es":1, "en":0, "reject":0, "probably reject": 1, "no decision":2, "accept":3}, inplace=True)

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,remarks,timespan,text
0,112_1,112,0,4,0,1,NaN,2014-07-05,"Me parece un trabajo interesante, con énfasis ..."
1,104_1,104,3,4,0,1,"Al trabajo le falta ""algo de"" practica. Ahora ...",2014-07-05,Correcciones a artículo: “Manejo de informació...
2,108_2,108,0,4,-1,1,NaN,2014-07-05,El artículo presenta un análisis teórico sobre...
3,74_1,74,3,4,0,1,NaN,2013-07-05,Los comentarios aparecen en el archivo adjunto
4,119_1,119,3,4,0,1,NaN,2014-07-05,El artículo describe una interesante aplicació...
...,...,...,...,...,...,...,...,...,...
76,30_2,30,0,4,-1,1,NaN,2010-07-05,Tema bien tratado técnicamente y en forma clar...
77,117_2,117,0,5,-2,1,NaN,2014-07-05,El trabajo presenta un metodo de detección de ...
78,21_2,21,3,4,-1,1,NaN,2010-07-05,Poca información sobre aspectos técnicos utili...
79,27_2,27,3,5,2,1,NaN,2010-07-05,"Un muy buen trabajo, presenta integración de v..."


In [84]:
test_df["review_id"]= test_df["review_id"].apply(returnrev)
test_df["timespan"] = test_df["timespan"].apply(returntime)
test_df

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,remarks,timespan,text
0,1,112,0,4,0,1,NaN,735325,"Me parece un trabajo interesante, con énfasis ..."
1,1,104,3,4,0,1,"Al trabajo le falta ""algo de"" practica. Ahora ...",735325,Correcciones a artículo: “Manejo de informació...
2,2,108,0,4,-1,1,NaN,735325,El artículo presenta un análisis teórico sobre...
3,1,74,3,4,0,1,NaN,734960,Los comentarios aparecen en el archivo adjunto
4,1,119,3,4,0,1,NaN,735325,El artículo describe una interesante aplicació...
...,...,...,...,...,...,...,...,...,...
76,2,30,0,4,-1,1,NaN,733865,Tema bien tratado técnicamente y en forma clar...
77,2,117,0,5,-2,1,NaN,735325,El trabajo presenta un metodo de detección de ...
78,2,21,3,4,-1,1,NaN,733865,Poca información sobre aspectos técnicos utili...
79,2,27,3,5,2,1,NaN,733865,"Un muy buen trabajo, presenta integración de v..."


In [85]:
def changeremarks(X):
    if pd.isna(X):
        return 0
    else:
        return 1
test_df["remarks_stat"] = test_df["remarks"].apply(changeremarks)
test_df.drop(["remarks"], axis=1)
test_df.head()

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,remarks,timespan,text,remarks_stat
0,1,112,0,4,0,1,NaN,735325,"Me parece un trabajo interesante, con énfasis ...",0
1,1,104,3,4,0,1,"Al trabajo le falta ""algo de"" practica. Ahora ...",735325,Correcciones a artículo: “Manejo de informació...,1
2,2,108,0,4,-1,1,NaN,735325,El artículo presenta un análisis teórico sobre...,0
3,1,74,3,4,0,1,NaN,734960,Los comentarios aparecen en el archivo adjunto,0
4,1,119,3,4,0,1,NaN,735325,El artículo describe una interesante aplicació...,0


In [86]:
def impp(X):
    if pd.isna(X):
        return -1
    else:
        return X
test_df["confidence"]=test_df["confidence"].apply(impp)

In [87]:
test_df

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,remarks,timespan,text,remarks_stat
0,1,112,0,4,0,1,NaN,735325,"Me parece un trabajo interesante, con énfasis ...",0
1,1,104,3,4,0,1,"Al trabajo le falta ""algo de"" practica. Ahora ...",735325,Correcciones a artículo: “Manejo de informació...,1
2,2,108,0,4,-1,1,NaN,735325,El artículo presenta un análisis teórico sobre...,0
3,1,74,3,4,0,1,NaN,734960,Los comentarios aparecen en el archivo adjunto,0
4,1,119,3,4,0,1,NaN,735325,El artículo describe una interesante aplicació...,0
...,...,...,...,...,...,...,...,...,...,...
76,2,30,0,4,-1,1,NaN,733865,Tema bien tratado técnicamente y en forma clar...,0
77,2,117,0,5,-2,1,NaN,735325,El trabajo presenta un metodo de detección de ...,0
78,2,21,3,4,-1,1,NaN,733865,Poca información sobre aspectos técnicos utili...,0
79,2,27,3,5,2,1,NaN,733865,"Un muy buen trabajo, presenta integración de v...",0


In [93]:
test_df.drop(["text", "remarks"], axis=1, inplace=True)

In [95]:
num2 = scaler.transform(test_df[["timespan", "paper_id"]])
test_out = np.hstack((num2, test_df.drop(["timespan", "paper_id"], axis=1) ))

In [97]:
log = LogisticRegression()
log.fit(x_out, y_train)
f1_score(log.predict(val_out), y_val, average="weighted")

np.float64(0.6437488546620708)

In [102]:

anothertest = pd.read_csv("testapepr.csv")

In [104]:
anothertest.head()

,review_id,paper_id,preliminary_decision,confidence,orientation,lan,remarks,timespan,text
0,112_1,112,reject,4,0,es,NaN,2014-07-05,"Me parece un trabajo interesante, con énfasis ..."
1,104_1,104,accept,4,0,es,"Al trabajo le falta ""algo de"" practica. Ahora ...",2014-07-05,Correcciones a artículo: “Manejo de informació...
2,108_2,108,reject,4,-1,es,NaN,2014-07-05,El artículo presenta un análisis teórico sobre...
3,74_1,74,accept,4,0,es,NaN,2013-07-05,Los comentarios aparecen en el archivo adjunto
4,119_1,119,accept,4,0,es,NaN,2014-07-05,El artículo describe una interesante aplicació...


In [105]:
preds_test = log.predict(test_out)

submission = pd.DataFrame({
    'id'   : anothertest['review_id'],
    'evaluation': preds_test
})

submission.to_csv('paperreview.csv', index=False)
print('submission.csv tersimpan!')
submission.head(10)

submission.csv tersimpan!


,id,evaluation
0,112_1,-2
1,104_1,1
2,108_2,-2
3,74_1,2
4,119_1,1
5,7_1,-2
6,164_2,2
7,110_1,2
8,10_2,2
9,30_3,-2
